[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/mosfet.ipynb)

# A MOSFET

This example computes a MOSFET, a metal–oxide–semiconductor field-effect transistor, by drift and diffusion: a cross-section of silicon in which a gate gathers a channel of electrons between two wells, and the current a drain voltage drives along it. The silicon is doped with acceptors, so that holes are its plentiful carriers, and two wells at its surface are doped with donors, the source and the drain, rich in electrons. Between them a thin oxide covers the surface, holding no carriers, and on the oxide sits the gate. A gate voltage pushes the holes away from the surface beneath it and draws electrons there, until, past a threshold, the electrons outnumber the holes in a thin layer joining source to drain: the channel. A drain voltage drives the electrons along it, and the current grows with the voltage at first; but the drain's voltage also weakens the gate's pull near the drain, the channel thins there until it pinches off, and past that the current hardly grows: it saturates, at a level the gate sets.

The algebra is the cross-section's plane in spacetime joined with two directions that have no metric, for the electrons and the holes, `Algebra("x+y+t-") * Algebra("n0p0")`; an electron is modelled as a unit along `n` with a negative charge, `mv.n.dual() - time`. The potentials at each vertex are one plane, which reads an electron's quasi-Fermi level by the regressive product, `potentials & electron`, and their drive, `-(gradients ^ potentials)`, pushes each carrier by what it holds, drift and diffusion in one. A carrier drifts along the force the drive puts on it, `drive.commutator(electron)`, the same in every direction, with no frame of axes to sum over. A corner's hat gradient is the side facing it joined with time and the amounts, over the same join closed by the step to the corner, so that the lines that compute a bar's edges compute the plane's triangles. The oxide is a material of its own, which carries the field's displacement at a third of silicon's permittivity and holds no carriers: the gate, which holds the electric potential at the oxide's top, makes a capacitor with the channel through Gauss's law alone.

In the notation of MOSFET device physics, with $V_{GS}$ and $V_{DS}$ the gate's and the drain's voltage, $C_{ox}$ the oxide's capacitance per area, $\mu_n$ the electrons' mobility and $W/L$ the channel's width over its length, the long-channel model holds the channel's charge per area at $Q_n = -C_{ox}(V_{GS} - V_T)$ above the threshold $V_T = V_{FB} + 2\phi_F + \sqrt{2\epsilon_s q N_A\,2\phi_F}/C_{ox}$, and gives the drain current $I_D = \mu_n C_{ox}\frac{W}{L}\big[(V_{GS} - V_T)V_{DS} - \tfrac12 V_{DS}^2\big]$ up to $V_{DS} = V_{GS} - V_T$, and beyond it, the channel pinched off, $I_D = \mu_n C_{ox}\frac{W}{2L}(V_{GS} - V_T)^2$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import math
from collections.abc import Iterator
from functools import reduce
from operator import and_

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_states
from examples.thermodynamics.semiconductor_devices import render, scenarios

# The cross-section's plane in spacetime, and the electrons and the holes a cell holds: each a direction of its own,
# degenerate, since nothing measures an amount against a length or a time.
ga = Algebra("x+y+t-") * Algebra("n0p0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# The planes reading a place across and up; the potentials, the electric potential along t and the carriers' chemical
# potentials along n and p; and their drive, a slope across the plane wedged with what each acts on.
Upright = ga.gatype.vector().derive.within(Algebra("x+y+"))                   # Vector+
Potentials = ga.gatype.vector().derive.within(Algebra("t-n0p0"))              # Vector+
Drive = ga.gatype(Upright ^ Potentials)                                       # Bivector+
# A unit of charge, along time; an electron, a unit of the electrons' count with a negative charge, and a hole, a unit
# of the holes' count with a positive one; and the two, what recombination takes.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
pairs = electron + hole                                                       # [] Point-
# The carriers; their counts, which their chemical potentials read, the planes that read each count off an amount,
# and the carriers' charges, along time.
carriers = stack((electron, hole))                                            # [carriers] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [carriers] Point-
counting = stack((mv.n, mv.p))                                                # [carriers] Plane+
charges = carriers - counts                                                   # [carriers] Point-

## The cross-section

The cross-section is silicon doped with acceptors, forty Debye lengths across and twenty deep, two wells of donors twenty times as dense at its surface, eight Debye lengths wide and four deep, and a gate on an oxide a Debye length thick between them, reaching a Debye length over each well. Lengths are in the substrate's Debye length, about 40 nm at 10¹⁶ acceptors a cubic centimetre, so that the cross-section is 1.6 µm across, the channel between the wells 1 µm long and the oxide 40 nm thick; energies and potentials are in the thermal energy, about 26 meV; densities are in the substrate's doping; and times are in the time an electron takes to diffuse across a Debye length. The source and the drain are contacts of metal reaching down through the oxide onto the wells beyond the gate, and the body is a contact along the bottom. The mesh is a grid of triangles, finest beneath the oxide, where the channel will be thinner than its finest rows, and across the wells' edges; each triangle is silicon or oxide.

In [ ]:
# Silicon at room temperature: its intrinsic density, a millionth of the substrate's doping; the electrons' and the
# holes' mobilities; the permittivities of silicon and of the oxide, a third of silicon's; and the rate constant of
# recombination, one over the doping times the minority carriers' lifetime.
intrinsic = 1e-6
mobilities = np.array([1.0, 1 / 3])
silicon_permittivity, oxide_permittivity = 1.0, 1 / 3
rate_constant = 1 / 2.0

In [ ]:
# The cross-section: its triangles, its vertices' places across and up, which triangles are oxide, the net doping at
# each vertex, the wells' donors less the substrate's acceptors, and the vertices of its contacts, the source, the
# drain, the body and the gate.
topology, coordinates, oxide, doping, contacts = scenarios.section()
vertices = mv(Upright, coordinates).dual().field()                            # Point-[vertices]
# Each triangle's material: silicon holds the carriers, and the oxide holds none and carries the field at a third of
# silicon's permittivity. The dopants' fixed charge lies along time.
semiconductor = mv.scalar(np.where(oxide, 0.0, 1.0)[:, None]).field()         # Scalar+[simplices]
permittivities = mv.scalar(np.where(oxide, oxide_permittivity, silicon_permittivity)[:, None]).field()   # Scalar+[simplices]
dopants = (time * doping).field()                                             # Point-[vertices]
render.draw_section(topology, vertices, semiconductor, dopants, contacts, scenarios.full_densities);

The picture shows the cross-section before any carrier moves, its silicon tinted by the carriers that would balance its dopants, electrons blue in the wells and holes red in the substrate, each at its full tint at the wells' doping and the substrate's; the oxide grey beneath the gate, dark, between the contacts' metal, and the body contact along the bottom. The silicon's surface beneath the oxide is drawn as a band of its own tint, wider than the channel will be, so that the channel shows.

## Drift, diffusion and the field

A field varies across each triangle by its corners' hats. A corner's hat gradient is the side facing it, joined with time and the amounts, which a field over the plane does not vary along, over the same join closed by the step to the corner; the drive across each triangle is then one map of the potentials at its corners, `driving`. Read backwards across the regressive product, the drive's adjugate gathers the fluxes in the triangles into what flows out of each vertex's cell. It pairs the amount on the left of the potentials, where the potentials read amounts from the left, as `potentials & electron` does, and an amount read from the right is its grade involute read from the left, `amount & potentials == potentials & amount.involute()`: the outflow is `driving.adjugate().involute()`. In an algebra of an odd count of directions, as here, the involute leaves an amount as it is; in one of an even count it turns its sign.

Each vertex's cell holds its share of the silicon about it, the oxide's triangles holding no carriers. Gauss's law, the field's outflow from each cell against the charge it holds, joins the carriers' balance along time, where that balance only repeats itself, and across the oxide's triangles, which hold no charge, the field's displacement carries the gate's pull to the silicon's surface. A steady state balances, in every cell, what the carriers' fluxes take out against what recombines; Newton's method solves it, each carrier's balance taken per carrier, so that carriers far scarcer than the plentiful ones, as the depletion regions hold them, weigh alike in the solve.

In [ ]:
corners = topology.incidence[-1, 0] * vertices                                # [corners] Point-[simplices]
# Time and the amounts, joined, which a field over the plane does not vary along.
unread = time & counts[0] & counts[1]                                         # [] Span-
# The corners of the side facing each corner, the first of them a step from the corner.
facing = corners[topology.complements[-1, 0]]                                 # [side corners, corners] Point-[simplices]
steps = corners - facing[0]                                                   # [corners] Point-[simplices]
# The side facing each corner joined with time and the amounts; over the same join closed by the step to the corner,
# it is the gradient that reads one from the side to the corner.
facets = reduce(and_, facing[1:] - facing[0], unread)                         # [corners] Plane-[simplices]
gradients = facets / (facets & steps)                                         # [corners] Plane+[simplices]
# The integrals weigh by each triangle's area, an ordinary number: the join of its edges from its first corner, with
# time and the amounts, over the factorial of its dimension. A vertex's cell is a share of the silicon about it, one
# over each triangle's count of corners.
joins = reduce(and_, corners[1:] - corners[0], unread)                        # Scalar-[simplices]
volumes = joins.norm() / math.factorial(topology.n_dim)                       # Scalar+[simplices]
corner_mean = topology.incidence[-1, 0].mean(axis=0)                          # Scalar+[simplices <- vertices]
cells = ~corner_mean * (volumes * semiconductor)                              # Scalar+[vertices]
# The drive across each triangle of the potentials on its corners: minus their slope, their corners' values on their
# hats' gradients, wedged with what each acts on; and what flows out of each vertex's cell of what flows in the
# triangles about it, the drive's map read backwards, its amounts read from the left.
slopes = (spdiag(gradients) * topology.incidence[-1, 0]).sum(axis=0)          # Plane+[simplices <- vertices]
driving = -(slopes ^ Potentials)                                              # (Drive+ <- Potentials+)[simplices <- vertices]
outflow = driving.adjugate().involute()                                       # (Point- <- Flux-)[vertices <- simplices]

In [ ]:
# A carrier drifts along the force a drive puts on it, and its flux is the carrier swept along that drift: each
# carrier's conductance per unit of its density, the same in every direction, in the silicon alone. A unit of charge
# swept so, times the permittivity, is the field's displacement.
conducting = (semiconductor * mobilities) * Drive.commutator(carriers).regressive(carriers)   # [carriers] (Flux- <- Drive+)[simplices]
displacement = permittivities * Drive.commutator(time).regressive(time)       # (Flux- <- Drive+)[simplices]


def carrier_densities(potentials: Potentials) -> Scalar:
    """The electrons and the holes per area at these potentials `[carriers] Scalar+[vertices]`: the intrinsic density
    times the exponential of each carrier's chemical potential, Boltzmann's distribution."""
    return intrinsic * (potentials & counts).exp()


def steady(potentials: Potentials, held: np.ndarray, contacts: Potentials) -> Iterator[Potentials]:
    """The steady potentials `Potentials+[vertices]` at each frame, from these: the vertices `held` at that frame's
    potentials of the contacts `[frames] Potentials+[held]`, each solved by Newton's steps until one changes no
    chemical potential by more than the tolerance."""
    holding = topology.reading(held, 0)                                       # Scalar+[held <- vertices]
    free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), held), 0)   # Scalar+[free <- vertices]
    for contact in contacts:
        # The contacts take their potentials; the rest start from where they were.
        potentials = ~free * (free * potentials) + ~holding * contact         # Potentials+[vertices]
        while True:
            densities = carrier_densities(potentials)                         # [carriers] Scalar+[vertices]
            drive = driving(potentials)                                       # Drive+[simplices]
            conductance = ((corner_mean * densities) * conducting).sum(axis=0)   # (Flux- <- Drive+)[simplices]
            # Recombination by mass action, pairs per area and time.
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2)   # Scalar+[vertices]
            # Gauss's law, along time: the field's outflow from each cell against the charge it holds.
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[vertices]
            gauss = outflow(volumes * displacement(drive)) - cells * charge  # Point-[vertices]
            # What the carriers' fluxes take out of each cell and what recombines there; along time this only repeats
            # the electrons' and the holes' balance, which Gauss's law completes there.
            residual = outflow(volumes * conductance(drive)) + cells * rates * pairs + gauss   # Point-[vertices]
            # Newton's map: the charge's capacitance in Gauss's law; the response of the carriers' fluxes and of the
            # field through the drive, and of the fluxes through the densities on each triangle, by each carrier's
            # flux per unit of its density, gathered into each cell's outflow; and recombination's, by the pairs' dyad.
            capacitance = spdiag(cells * (densities * -charges * (Potentials & counts)).sum(axis=0))
            through_drive = spdiag(volumes * (conductance + displacement))(driving)
            through_densities = (spdiag(volumes * conducting(drive)) * corner_mean
                                 * spdiag(densities * (Potentials & counts))).sum(axis=0)
            recombining = spdiag(cells * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs)))
            newton = capacitance + recombining + outflow(through_drive + through_densities)
            # Each carrier's balance taken per carrier, over its density, the charge's as it is.
            per_carrier = spdiag(time * (mv.t & Point) + (counts * (counting & Point) / densities).sum(axis=0))
            solving = free * per_carrier(newton) * ~free                      # (Point- <- Potentials+)[free <- free]
            change = -(~free * solving.solve(free * per_carrier(residual)))   # Potentials+[vertices]
            # Newton's step scaled down at each vertex where it would change the chemical potentials there by much
            # more than the limit: the densities follow them exponentially, and a full step from far off overshoots by
            # powers of e. The residual is linear in the electric potential, whose change needs no holding back.
            changes = (counts & change).abs().sum(axis=0)                     # Scalar+[vertices]
            potentials = potentials + change / (1 + changes / limit)
            if changes.max() < tolerance:
                break
        yield potentials

## The channel and its current

The contacts hold the potentials at their vertices. The source, the drain and the body are ohmic, held at neutral silicon of their doping, its Fermi level lowered by their voltage, none at the source and the body. The gate's metal holds the electric potential at the oxide's top, its work function the substrate's, so that at no gate voltage the bands lie flat, and its carriers' potentials idle, since the oxide holds none. The gate ramps at a small drain voltage until a channel joins source to drain; then the drain sweeps up at the first of three gate voltages, 1.8 V, the gate steps to 2.2 V and the drain sweeps back down, and the gate steps to 2.6 V for a last sweep up. Each frame is a steady state, the first found from neutral silicon. The drain's voltage is the rise of the electrons' quasi-Fermi level from the drain to the source, and its current is the charge the carriers' fluxes bring into the source's cells, against what flows out of them, `mv.t & outflow(volumes * flux)` there. This cell takes about half a minute, most of it solving for the steady states.

In [ ]:
# Newton's steps: how far one may change a chemical potential, in thermal units, and how little the last one changes
# them; the frames' duration in milliseconds.
limit, tolerance = 2.0, 1e-6
frame_duration_ms = 40
# The gate's and the drain's voltage at each frame and the output curve each lies on, minus one where it lies on none;
# the volt in thermal units, at 300 K, and the output curves' windows.
gates, drains, curves = scenarios.schedule()                                  # [frames], [frames], [frames]
volt = 1 / 0.02585
largest_drain, current_window = scenarios.largest_drain, scenarios.drain_current_window

In [ ]:
# Neutral silicon with its Fermi level at nothing: each carrier's electrochemical potential is nothing, so that the
# electric potential is the electrons' chemical potential. The gate's metal holds the substrate's electric potential at
# no gate voltage.
chemical = scenarios.slabs(doping)                                            # [carriers, vertices]
substrate = scenarios.slabs(np.array(-1.0))[0]
source, drain, body, gate = contacts
on_gate = np.isin(np.arange(topology.n_vertices), gate)                       # [vertices]
neutral = (mv.t * np.where(on_gate, substrate, chemical[0]) + mv.n * np.where(on_gate, 0.0, chemical[0])
           + mv.p * np.where(on_gate, 0.0, chemical[1])).field()             # Potentials+[vertices]
# The contacts at each frame: the drain's voltage and the gate's raise the electric potential they hold.
held = np.concatenate(contacts)                                               # [held]
raised = np.concatenate([np.zeros((len(gates), len(source))), np.repeat(drains[:, None], len(drain), 1),
                         np.zeros((len(gates), len(body))), np.repeat(gates[:, None], len(gate), 1)], axis=-1)
wired = topology.reading(held, 0) * neutral + (mv.t * raised).field()         # [frames] Potentials+[held]


def terminal(potentials: Potentials) -> tuple[Scalar, Scalar]:
    """The drain's voltage at these potentials `[] Scalar+`, the rise of the electrons' quasi-Fermi level from the
    drain to the source, and its current `[] Scalar+`, the charge the carriers' fluxes bring into the source's cells."""
    levels = potentials & electron                                            # Scalar+[vertices]
    rise = (topology.reading(source, 0) * levels).sites.mean() - (topology.reading(drain, 0) * levels).sites.mean()
    conductance = ((corner_mean * carrier_densities(potentials)) * conducting).sum(axis=0)   # (Flux- <- Drive+)[simplices]
    outflowing = outflow(volumes * conductance(driving(potentials)))          # Point-[vertices]
    return rise, -(topology.reading(source, 0) * (mv.t & outflowing)).sites.sum()


switched = show_states(steady(neutral, held, wired), render.animate_switching, topology, vertices, semiconductor,
                       contacts, carrier_densities, terminal, scenarios.full_densities, curves, largest_drain,
                       current_window, volt, name="mosfet", duration_ms=frame_duration_ms)

The animation shows the cross-section beside the drain's current against its voltage, in volts, an output curve traced for each gate voltage, lightest at the lowest. As the gate ramps, the holes leave the surface beneath it and the depletion region under the gate deepens, white; past the threshold, near 1.4 V, electrons gather in a thin layer under the oxide, blue, and join the wells. As the drain's voltage rises the current grows, in proportion at first; the drain's depletion region widens, and the channel thins toward the drain until it pinches off there, fading, and the current saturates: each output curve levels off, the higher the higher the gate. The curves still climb a little once saturated, the pinched-off stretch of the channel lengthening toward the source as the drain's voltage rises.

In [ ]:
# checks
# The electrons per area of the surface between two places across, all of them beneath it.
across = mv.x & vertices                                                      # Scalar+[vertices]


def sheet(potentials: Potentials, left: float, right: float) -> float:
    window = topology.reading(np.flatnonzero((across > left) & (across < right)), 0)
    return (window * (cells * carrier_densities(potentials)[0])).sites.sum().to_array() / (right - left)


voltages, currents = (stack(readings).to_array() for readings in zip(*(terminal(state) for state in switched)))
# Past the threshold, where the line through the ramp's last two frames meets nothing, the channel's electrons per
# area under the gate's middle grow at the oxide's capacitance per area; well below it the drain's current is
# negligible.
ramp = np.arange(scenarios.n_ramp_frames)
middle = np.array([sheet(switched[frame], -4.0, 4.0) for frame in ramp])      # [ramp]
threshold = gates[ramp[-1]] - middle[-1] * (gates[ramp[-1]] - gates[ramp[-2]]) / (middle[-1] - middle[-2])
strong = ramp[gates[ramp] > threshold + 6]
growth = np.diff(middle[strong]) / np.diff(gates[strong]) / (oxide_permittivity / scenarios.oxide_thickness)
assert len(strong) > 2 and np.all(np.abs(growth - 1) < 0.2)
below = ramp[gates[ramp] < threshold - 10]
assert len(below) > 0 and np.abs(currents[below]).max() < 1e-3 * currents[ramp[-1]]
# On each sweep the current is linear in the drain's voltage while it is small, and saturates once it is large.
for curve in range(len(scenarios.gate_voltages)):
    order = np.flatnonzero(curves == curve)[np.argsort(voltages[curves == curve])]
    conductance = currents[order[:2]] / voltages[order[:2]]
    assert abs(conductance[1] / conductance[0] - 1) < 0.15
    assert np.all(np.diff(currents[order[-3:]]) / np.diff(voltages[order[-3:]]) < conductance[0] / 6)
# At the first gate voltage the channel near the drain holds as many electrons as near the source while the drain's
# voltage is small, and far fewer once the current has saturated: it pinches off near the drain.
on_first = np.flatnonzero(curves == 0)
linear, saturated = on_first[voltages[on_first].argmin()], on_first[voltages[on_first].argmax()]
assert sheet(switched[linear], 8.0, 11.0) > 0.8 * sheet(switched[linear], -11.0, -8.0)
assert sheet(switched[saturated], 8.0, 11.0) < 0.2 * sheet(switched[saturated], -11.0, -8.0)